# Skin Lesion Classification — HAM10000

**Dataset:** HAM10000 — 10,015 dermoscopic images, 7 classes  
**Task:** Multi-class classification (supervised learning)  
**Stack:** PyTorch, torchvision, scikit-learn, grad-cam  
**Environment:** Google Colab (GPU T4/A100)

| Class | Code | Description |
|-------|------|-------------|
| Melanocytic nevus | nv | Benign mole |
| Melanoma | mel | Malignant tumor |
| Benign keratosis | bkl | Benign keratosis |
| Basal cell carcinoma | bcc | Basal cell carcinoma |
| Actinic keratosis | akiec | Pre-cancerous lesion |
| Vascular lesion | vasc | Vascular lesion |
| Dermatofibroma | df | Dermatofibroma |

## Problem Description

Skin cancer is one of the most common cancers worldwide. Early and accurate diagnosis significantly improves patient outcomes. Dermoscopy — a non-invasive imaging technique — allows detailed visualization of skin lesions, but manual classification by dermatologists is time-consuming and subject to inter-observer variability.

**Goal:** Build a multi-class image classifier that automatically assigns a dermoscopic image to one of 7 diagnostic categories, ranging from benign moles to malignant melanoma.

**Type of learning:** Supervised multi-class classification. Each image has a confirmed ground-truth label from clinical evaluation, histopathology, or expert consensus (follow-up).

**Key challenge:** Severe class imbalance — *melanocytic nevus* (benign mole) accounts for ~67% of all samples, while rare classes such as dermatofibroma and vascular lesion represent less than 2% each. A naive model that always predicts *nevus* achieves ~67% accuracy while being clinically useless.

---

## Solution Pipeline

| Step | Description |
|------|-------------|
| 1. EDA | Understand class distribution, patient demographics, and visual image characteristics |
| 2. Data preparation | Stratified train / val / test split (70/15/15); class-balanced sampling via WeightedRandomSampler |
| 3. Transforms | Image augmentation for training; normalization for all splits |
| 4. Baseline model | Train a simple 4-layer CNN from scratch to establish a performance lower bound |
| 5. Transfer learning | Fine-tune ResNet18 (pre-trained on ImageNet) in two phases: frozen backbone → full fine-tune |
| 6. Error analysis | Confusion matrix, Grad-CAM attention maps, per-class diagnostics |
| 7. Comparison | Side-by-side evaluation of both models: Accuracy, Weighted F1, AUC-ROC |

## Stage 1 — Environment Setup

In [ ]:
# Install dependencies
!pip install grad-cam -q

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from PIL import Image
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import torchvision
import torchvision.transforms as transforms
from torchvision import models

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, f1_score
)

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

os.makedirs('plots', exist_ok=True)

In [ ]:
# ─── Global configuration ─────────────────────────────────────────────────────
DATASET_PATH    = 'data'
CSV_PATH        = os.path.join(DATASET_PATH, 'HAM10000_metadata.csv')
IMG_DIRS        = [
    os.path.join(DATASET_PATH, 'HAM10000_images_part_1'),
    os.path.join(DATASET_PATH, 'HAM10000_images_part_2'),
]

IMG_SIZE        = 224   # standard input size for EfficientNet/ResNet
BATCH_SIZE      = 32
NUM_CLASSES     = 7
EPOCHS_FROZEN   = 5     # epochs with frozen backbone
EPOCHS_FINETUNE = 15    # epochs for full fine-tuning
LR_FROZEN       = 1e-3
LR_FINETUNE     = 3e-5

CLASS_NAMES = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
CLASS_LABELS = {
    'akiec': 'Actinic keratosis',
    'bcc':   'Basal cell carcinoma',
    'bkl':   'Benign keratosis',
    'df':    'Dermatofibroma',
    'mel':   'Melanoma',
    'nv':    'Melanocytic nevus',
    'vasc':  'Vascular lesion',
}

## Stage 2 — Exploratory Data Analysis (EDA)

### Dataset Structure

**HAM10000** ("Human Against Machine with 10000 training images") is a large public collection of dermoscopic images published in 2018. It is one of the standard benchmarks for skin lesion classification and was used in the ISIC 2018 challenge.

**Images:** 10,015 JPEG files (450×600 px), stored across two directories:
- `HAM10000_images_part_1`
- `HAM10000_images_part_2`

All images are resized to **224×224 px** during loading (standard input size for ImageNet-pretrained models).

**Metadata CSV columns:**

| Column | Type | Description |
|--------|------|-------------|
| `image_id` | string | Unique identifier — matches the image filename |
| `dx` | string | **Ground truth diagnosis** (one of 7 class codes) |
| `dx_type` | string | Confirmation method: `histo` (histopathology), `follow_up`, `consensus`, `confocal` |
| `age` | float | Patient age in years (may be `NaN`) |
| `sex` | string | Patient sex: `male`, `female`, or `unknown` |
| `localization` | string | Body location of the lesion (e.g., back, face, trunk) |

**Data split:** Stratified 70 / 15 / 15 (train / val / test). Stratification ensures that class proportions are preserved in every split — crucial given the severe class imbalance.

In [ ]:
df = pd.read_csv(CSV_PATH)
print(f"Dataset size: {len(df)} images")
print(f"Columns: {list(df.columns)}")
df.head()

In [ ]:
# Build image_id -> file path mapping
img_path_map = {}
for img_dir in IMG_DIRS:
    if not os.path.exists(img_dir):
        continue
    for fname in os.listdir(img_dir):
        img_id = os.path.splitext(fname)[0]
        img_path_map[img_id] = os.path.join(img_dir, fname)

df['path'] = df['image_id'].map(img_path_map)
missing = df['path'].isna().sum()
print(f"Images found: {len(df) - missing}/{len(df)}")
df = df.dropna(subset=['path']).reset_index(drop=True)

# Numeric label
df['label'] = df['dx'].map({c: i for i, c in enumerate(CLASS_NAMES)})
df[['image_id', 'dx', 'label', 'path']].head()

In [ ]:
# Class distribution
counts = df['dx'].value_counts().reindex(CLASS_NAMES)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = sns.color_palette('Set2', NUM_CLASSES)
axes[0].bar([CLASS_LABELS[c] for c in counts.index], counts.values, color=colors)
axes[0].set_title('Class distribution (absolute)')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Number of images')
axes[0].tick_params(axis='x', rotation=45)
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 30, str(v), ha='center', fontsize=9)

axes[1].pie(
    counts.values,
    labels=[CLASS_LABELS[c] for c in counts.index],
    autopct='%1.1f%%',
    colors=colors,
    startangle=140
)
axes[1].set_title('Class distribution (percentage)')

plt.suptitle('Class Imbalance in HAM10000', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nStatistics:")
for cls, cnt in counts.items():
    print(f"  {CLASS_LABELS[cls]:30s}: {cnt:5d} ({cnt/len(df)*100:.1f}%)")

In [ ]:
# One sample image per class
fig = plt.figure(figsize=(16, 10))
gs = gridspec.GridSpec(2, 4, figure=fig)

for idx, cls in enumerate(CLASS_NAMES):
    samples = df[df['dx'] == cls].sample(1, random_state=SEED)
    img = Image.open(samples.iloc[0]['path']).resize((224, 224))
    ax = fig.add_subplot(gs[idx // 4, idx % 4])
    ax.imshow(img)
    ax.set_title(f"{CLASS_LABELS[cls]}\n({cls})", fontsize=9)
    ax.axis('off')

fig.suptitle('One sample per class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('sample_images.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Additional EDA: patient age and sex
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

df['age'].dropna().hist(ax=axes[0], bins=20, color='steelblue', edgecolor='white')
axes[0].set_title('Patient age distribution')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Count')

sex_counts = df['sex'].value_counts()
axes[1].bar(sex_counts.index, sex_counts.values, color=['#4C72B0', '#DD8452'])
axes[1].set_title('Sex distribution')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

## Stage 3 — Data Transforms & DataLoaders

### Data Transforms & Augmentation

**Why augmentation?**  
The training set (~7,000 images) is small relative to model capacity. Augmentation artificially expands the effective dataset by generating plausible image variants, reducing overfitting and improving generalization.

Different transforms are applied at different stages:

| Transform | Train | Val / Test | Rationale |
|-----------|:-----:|:----------:|-----------|
| `Resize(224×224)` | ✅ | ✅ | Required input size for ResNet18; reduces memory footprint |
| `RandomHorizontalFlip` | ✅ | ❌ | Lesions have no natural orientation — horizontal mirroring is valid |
| `RandomVerticalFlip` | ✅ | ❌ | Same rationale as horizontal flip |
| `RandomRotation(±20°)` | ✅ | ❌ | A lesion is diagnostically the same at any rotation angle |
| `ColorJitter(b=0.2, c=0.2, s=0.2, h=0.1)` | ✅ | ❌ | Simulates lighting variation and differences between imaging devices across clinics |
| `ToTensor` | ✅ | ✅ | Converts PIL image to float tensor in [0, 1] |
| `Normalize(ImageNet µ, σ)` | ✅ | ✅ | Matches the input distribution expected by ImageNet-pretrained weights |

**Class imbalance handling — WeightedRandomSampler:**  
Each training sample is assigned a weight inversely proportional to its class frequency. The sampler draws batches according to these weights, so each mini-batch contains a more balanced class distribution — without duplicating images or modifying the dataset on disk. Validation and test sets remain untouched to reflect the true class distribution during evaluation.

In [ ]:
# Stratified split: 70% train / 15% val / 15% test
train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df['label'], random_state=SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df['label'], random_state=SEED
)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

In [ ]:
# ImageNet normalization statistics
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

In [ ]:
class SkinDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['path']).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, int(row['label'])


train_dataset = SkinDataset(train_df, transform=train_transform)
val_dataset   = SkinDataset(val_df,   transform=val_transform)
test_dataset  = SkinDataset(test_df,  transform=val_transform)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

In [ ]:
# Weighted sampler to handle class imbalance
label_counts = Counter(train_df['label'].values)
class_weights = {cls: 1.0 / count for cls, count in label_counts.items()}
sample_weights = [class_weights[label] for label in train_df['label'].values]

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,    num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,    num_workers=2, pin_memory=True)

print("DataLoaders created.")

# Sanity check
imgs, labels = next(iter(train_loader))
print(f"Batch shape: {imgs.shape}, labels: {labels[:8].tolist()}")

## Stage 4 — Baseline CNN

### Baseline CNN — Architecture & Design Choices

A simple convolutional network trained **from scratch** (random weight initialization, no pre-trained weights). Its purpose is to establish a performance **lower bound** and demonstrate the benefit of transfer learning.

```
Input:  3 × 224 × 224  (RGB image)

Feature extractor:
  Block 1:  Conv2d(3→32,   3×3, pad=1) → BatchNorm2d → ReLU → MaxPool(2×2)   → 32×112×112
  Block 2:  Conv2d(32→64,  3×3, pad=1) → BatchNorm2d → ReLU → MaxPool(2×2)   → 64×56×56
  Block 3:  Conv2d(64→128, 3×3, pad=1) → BatchNorm2d → ReLU → MaxPool(2×2)   → 128×28×28
  Block 4:  Conv2d(128→256,3×3, pad=1) → BatchNorm2d → ReLU → AdaptiveAvg(4×4) → 256×4×4

Classifier head:
  Flatten → 4096
  Linear(4096→512) → ReLU → Dropout(0.4)
  Linear(512→7)

Output: 7 class logits
Total parameters: ~4.7 M
```

**Design choices:**

| Choice | Rationale |
|--------|-----------|
| BatchNorm after every conv layer | Stabilizes gradient flow, allows higher learning rates, acts as mild regularizer |
| Doubling channels (32→64→128→256) | Each block captures increasingly abstract features |
| AdaptiveAvgPool(4×4) in last block | Avoids excessive spatial reduction while still compressing the feature map |
| Dropout(0.4) before final FC | Regularization — prevents co-adaptation of neurons and reduces overfitting |
| Adam + ReduceLROnPlateau | Adaptive LR optimizer; scheduler halves LR if val loss stagnates for 3 epochs |

In [ ]:
class BaselineCNN(nn.Module):
    """Simple 4-layer CNN trained from scratch as a baseline."""
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.AdaptiveAvgPool2d(4),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


baseline_model = BaselineCNN().to(DEVICE)
total_params = sum(p.numel() for p in baseline_model.parameters())
print(f"Baseline CNN parameters: {total_params:,}")

In [ ]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += imgs.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        total_loss += loss.item() * imgs.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += imgs.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    return total_loss / total, correct / total, all_preds, all_labels


def train_model(model, train_loader, val_loader, optimizer, scheduler, criterion, epochs, model_name='model'):
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_val_acc = 0.0

    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, criterion)
        vl_loss, vl_acc, _, _ = evaluate(model, val_loader, criterion)

        if scheduler:
            scheduler.step(vl_loss)

        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc)
        history['val_loss'].append(vl_loss)
        history['val_acc'].append(vl_acc)

        if vl_acc > best_val_acc:
            best_val_acc = vl_acc
            torch.save(model.state_dict(), f'best_{model_name}.pth')

        print(f"Epoch {epoch:3d}/{epochs} | "
              f"Train Loss: {tr_loss:.4f} Acc: {tr_acc:.4f} | "
              f"Val Loss: {vl_loss:.4f} Acc: {vl_acc:.4f}"
              + (" * best" if vl_acc == best_val_acc else ""))

    print(f"\nBest val accuracy: {best_val_acc:.4f}")
    return history

In [ ]:
# Train baseline
criterion_baseline = nn.CrossEntropyLoss()
optimizer_baseline = optim.Adam(baseline_model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler_baseline = optim.lr_scheduler.ReduceLROnPlateau(optimizer_baseline, patience=3, factor=0.5)

print("=" * 60)
print("Training Baseline CNN")
print("=" * 60)
baseline_history = train_model(
    baseline_model, train_loader, val_loader,
    optimizer_baseline, scheduler_baseline, criterion_baseline,
    epochs=10, model_name='baseline'
)

In [ ]:
def plot_training_history(history, title='Baseline CNN'):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(history['train_loss'], label='Train', linewidth=2)
    axes[0].plot(history['val_loss'],   label='Val',   linewidth=2)
    axes[0].set_title(f'{title} — Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    axes[1].plot([a * 100 for a in history['train_acc']], label='Train', linewidth=2)
    axes[1].plot([a * 100 for a in history['val_acc']],   label='Val',   linewidth=2)
    axes[1].set_title(f'{title} — Accuracy')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy (%)')
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'{title.replace(" ", "_")}_history.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_training_history(baseline_history, 'Baseline CNN')

## Stage 5 — Transfer Learning (ResNet18)

### Transfer Learning — ResNet18 Architecture & Training Strategy

**Why transfer learning?**  
ResNet18 pre-trained on ImageNet has already learned general visual features — edges, textures, color gradients, and shapes. Although dermoscopic images differ from natural photos, low-level features transfer remarkably well. This means we need far fewer epochs and far less data to reach strong performance.

**Modified architecture:**

```
ImageNet ResNet18 backbone (frozen in Phase 1, unfrozen in Phase 2):
  Conv2d(3→64) → BN → ReLU → MaxPool
  Layer1: 2× BasicBlock  (64 channels,  stride=1)
  Layer2: 2× BasicBlock  (128 channels, stride=2)
  Layer3: 2× BasicBlock  (256 channels, stride=2)
  Layer4: 2× BasicBlock  (512 channels, stride=2)  ← Grad-CAM target layer
  AdaptiveAvgPool(1×1)  →  512-dim feature vector

Custom classifier head (replaces original FC(512→1000)):
  Linear(512→256) → ReLU → Dropout(0.3)
  Linear(256→7)

Total parameters: ~11.2 M
```

**Two-phase training strategy:**

| Phase | Backbone | Head | Epochs | LR (backbone) | LR (head) | Scheduler |
|-------|:--------:|:----:|:------:|:-------------:|:---------:|-----------|
| 1 — Frozen | ❄️ Frozen | 🔥 Train | 5 | — | 1e-3 | ReduceLROnPlateau |
| 2 — Fine-tune | 🔥 Unfrozen | 🔥 Train | 15 | 3e-5 | 1e-3 | CosineAnnealingLR |

- **Phase 1** trains only the classifier head. The backbone is frozen — its weights are not updated. This prevents destroying the pre-trained features before the head can produce stable gradients.
- **Phase 2** unfreezes the entire network and applies **differential learning rates**: the backbone uses a very small LR (3e-5) to gently adapt its weights, while the head uses the same LR as Phase 1. `CosineAnnealingLR` smoothly decays the LR to zero over 15 epochs, avoiding oscillation near the optimum.

In [ ]:
def build_resnet18(num_classes=NUM_CLASSES, freeze_backbone=True):
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False

    # Replace the final classifier
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Linear(in_features, 256),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(256, num_classes),
    )
    return model


resnet_model = build_resnet18(freeze_backbone=True).to(DEVICE)

trainable = sum(p.numel() for p in resnet_model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in resnet_model.parameters())
print(f"Parameters: {total:,} total, {trainable:,} trainable ({trainable/total*100:.1f}%)")

In [ ]:
# Phase 1: train only the classifier head
optimizer_frozen = optim.Adam(
    filter(lambda p: p.requires_grad, resnet_model.parameters()),
    lr=LR_FROZEN, weight_decay=1e-4
)
scheduler_frozen = optim.lr_scheduler.ReduceLROnPlateau(optimizer_frozen, patience=2, factor=0.5)
criterion_resnet = nn.CrossEntropyLoss()

print("=" * 60)
print("Phase 1: frozen backbone")
print("=" * 60)
history_frozen = train_model(
    resnet_model, train_loader, val_loader,
    optimizer_frozen, scheduler_frozen, criterion_resnet,
    epochs=EPOCHS_FROZEN, model_name='resnet_frozen'
)

In [ ]:
# Phase 2: unfreeze the entire backbone and fine-tune
for param in resnet_model.parameters():
    param.requires_grad = True

# Differential learning rates: lower LR for backbone, higher LR for head
optimizer_ft = optim.AdamW([
    {'params': list(resnet_model.parameters())[:-4], 'lr': LR_FINETUNE},
    {'params': list(resnet_model.parameters())[-4:], 'lr': LR_FROZEN},
], weight_decay=1e-4)
scheduler_ft = optim.lr_scheduler.CosineAnnealingLR(optimizer_ft, T_max=EPOCHS_FINETUNE)

print("=" * 60)
print("Phase 2: full fine-tuning")
print("=" * 60)
history_ft = train_model(
    resnet_model, train_loader, val_loader,
    optimizer_ft, scheduler_ft, criterion_resnet,
    epochs=EPOCHS_FINETUNE, model_name='resnet_finetuned'
)

# Combine history from both phases
combined_history = {
    k: history_frozen[k] + history_ft[k] for k in history_frozen
}
plot_training_history(combined_history, 'ResNet18 Transfer Learning')

In [ ]:
# ─── Comparative training curves: Baseline CNN vs ResNet18 ────────────────────
# Note: x-axes differ in length (10 vs 20 epochs) — shown on separate subplots.
# The vertical dashed line marks the moment the ResNet18 backbone was unfrozen.

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── Loss ──────────────────────────────────────────────────────────────────────
ep_base   = range(1, len(baseline_history['train_loss']) + 1)
ep_resnet = range(1, len(combined_history['train_loss']) + 1)

axes[0].plot(ep_base,   baseline_history['train_loss'], label='Baseline — Train',  color='#4C72B0', linewidth=2)
axes[0].plot(ep_base,   baseline_history['val_loss'],   label='Baseline — Val',    color='#4C72B0', linewidth=2, linestyle='--')
axes[0].plot(ep_resnet, combined_history['train_loss'], label='ResNet18 — Train',  color='#DD8452', linewidth=2)
axes[0].plot(ep_resnet, combined_history['val_loss'],   label='ResNet18 — Val',    color='#DD8452', linewidth=2, linestyle='--')
axes[0].axvline(x=EPOCHS_FROZEN, color='gray', linestyle=':', linewidth=1.5, label=f'ResNet18: backbone unfrozen (ep {EPOCHS_FROZEN})')
axes[0].set_title('Training Loss — Baseline vs ResNet18', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

# ── Accuracy ──────────────────────────────────────────────────────────────────
axes[1].plot(ep_base,   [a * 100 for a in baseline_history['train_acc']], label='Baseline — Train',  color='#4C72B0', linewidth=2)
axes[1].plot(ep_base,   [a * 100 for a in baseline_history['val_acc']],   label='Baseline — Val',    color='#4C72B0', linewidth=2, linestyle='--')
axes[1].plot(ep_resnet, [a * 100 for a in combined_history['train_acc']], label='ResNet18 — Train',  color='#DD8452', linewidth=2)
axes[1].plot(ep_resnet, [a * 100 for a in combined_history['val_acc']],   label='ResNet18 — Val',    color='#DD8452', linewidth=2, linestyle='--')
axes[1].axvline(x=EPOCHS_FROZEN, color='gray', linestyle=':', linewidth=1.5, label=f'ResNet18: backbone unfrozen (ep {EPOCHS_FROZEN})')
axes[1].set_title('Validation Accuracy — Baseline vs ResNet18', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

plt.suptitle('Model Comparison: Training Dynamics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('comparison_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## Stage 6 — Error Analysis (Confusion Matrix + Grad-CAM)

In [ ]:
# Load the best checkpoint
resnet_model.load_state_dict(torch.load('best_resnet_finetuned.pth', map_location=DEVICE))

_, test_acc, test_preds, test_labels = evaluate(resnet_model, test_loader, criterion_resnet)
print(f"Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")

# Confusion Matrix
cm = confusion_matrix(test_labels, test_preds)
cm_normalized = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[0])
axes[0].set_title('Confusion Matrix (counts)')
axes[0].set_xlabel('Predicted class')
axes[0].set_ylabel('True class')

sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='YlOrRd',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[1],
            vmin=0, vmax=1)
axes[1].set_title('Confusion Matrix (row-normalized)')
axes[1].set_xlabel('Predicted class')
axes[1].set_ylabel('True class')

plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

# Target layer for Grad-CAM in ResNet18 — last convolutional block
target_layer = [resnet_model.layer4[-1]]

cam = GradCAM(model=resnet_model, target_layers=target_layer)

def get_gradcam_image(img_path, true_label, pred_label, transform):
    """Returns original image and Grad-CAM overlay as numpy arrays."""
    img_pil = Image.open(img_path).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
    img_np  = np.array(img_pil) / 255.0
    img_tensor = transform(img_pil).unsqueeze(0).to(DEVICE)

    targets = [ClassifierOutputTarget(pred_label)]
    grayscale_cam = cam(input_tensor=img_tensor, targets=targets)[0]
    visualization = show_cam_on_image(img_np.astype(np.float32), grayscale_cam, use_rgb=True)
    return img_np, visualization

print("Grad-CAM ready.")

In [ ]:
# Grad-CAM visualization: correct and incorrect predictions side by side
resnet_model.eval()

samples_to_show = []
for cls_idx in range(NUM_CLASSES):
    cls_mask = (np.array(test_labels) == cls_idx)
    candidates = test_df[cls_mask].copy()
    candidates['pred'] = np.array(test_preds)[cls_mask]
    correct   = candidates[candidates['pred'] == cls_idx]
    incorrect = candidates[candidates['pred'] != cls_idx]

    if len(correct) > 0:
        row = correct.sample(1, random_state=SEED).iloc[0]
        samples_to_show.append((row['path'], cls_idx, int(row['pred']), True))
    if len(incorrect) > 0:
        row = incorrect.sample(1, random_state=SEED).iloc[0]
        samples_to_show.append((row['path'], cls_idx, int(row['pred']), False))

n_show = min(len(samples_to_show), 8)
samples_to_show = samples_to_show[:n_show]

fig, axes = plt.subplots(n_show, 2, figsize=(8, n_show * 3))
if n_show == 1:
    axes = [axes]

for i, (path, true_lbl, pred_lbl, is_correct) in enumerate(samples_to_show):
    orig, cam_viz = get_gradcam_image(path, true_lbl, pred_lbl, val_transform)
    status = 'Correct' if is_correct else 'Wrong'
    color  = 'green' if is_correct else 'red'

    axes[i][0].imshow(orig)
    axes[i][0].set_title(f"{status} | True: {CLASS_NAMES[true_lbl]}", color=color, fontsize=9)
    axes[i][0].axis('off')

    axes[i][1].imshow(cam_viz)
    axes[i][1].set_title(f"Pred: {CLASS_NAMES[pred_lbl]}  (Grad-CAM)", fontsize=9)
    axes[i][1].axis('off')

plt.suptitle('Grad-CAM: what the network focuses on', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('gradcam_results.png', dpi=150, bbox_inches='tight')
plt.show()

## Stage 7 — Final Evaluation & Model Comparison

In [ ]:
# ─── Evaluate Baseline CNN ────────────────────────────────────────────────────
baseline_model.load_state_dict(torch.load('best_baseline.pth', map_location=DEVICE))
_, base_acc, base_preds, base_labels = evaluate(baseline_model, test_loader, criterion_baseline)
base_f1 = f1_score(base_labels, base_preds, average='weighted')

# ─── Evaluate ResNet18 ────────────────────────────────────────────────────────
resnet_model.load_state_dict(torch.load('best_resnet_finetuned.pth', map_location=DEVICE))
_, resnet_acc, resnet_preds, resnet_labels = evaluate(resnet_model, test_loader, criterion_resnet)
resnet_f1 = f1_score(resnet_labels, resnet_preds, average='weighted')

print("=" * 55)
print(f"{'Model':<20} {'Accuracy':>10} {'Weighted F1':>13}")
print("-" * 55)
print(f"{'Baseline CNN':<20} {base_acc*100:>9.2f}% {base_f1:>13.4f}")
print(f"{'ResNet18 FT':<20} {resnet_acc*100:>9.2f}% {resnet_f1:>13.4f}")
print("=" * 55)

In [ ]:
# ─── Overall performance comparison bar chart (Accuracy + Weighted F1) ────────
metrics_names  = ['Test Accuracy', 'Weighted F1']
baseline_vals  = [base_acc,   base_f1]
resnet_vals    = [resnet_acc, resnet_f1]

x     = np.arange(len(metrics_names))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
bars1 = ax.bar(x - width/2, baseline_vals, width, label='Baseline CNN', color='#4C72B0', alpha=0.87)
bars2 = ax.bar(x + width/2, resnet_vals,   width, label='ResNet18 FT',  color='#DD8452', alpha=0.87)

ax.set_xticks(x)
ax.set_xticklabels(metrics_names, fontsize=12)
ax.set_ylim(0, 1.12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Overall Performance: Baseline CNN vs ResNet18', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.015,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold', color='#4C72B0')
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.015,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold', color='#DD8452')

plt.tight_layout()
plt.savefig('accuracy_f1_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Detailed per-class report
label_names = [CLASS_LABELS[c] for c in CLASS_NAMES]
print("\nResNet18 — Classification Report:")
print(classification_report(resnet_labels, resnet_preds, target_names=label_names))

In [ ]:
# Per-class F1-score — both models side by side
from sklearn.metrics import f1_score

base_f1_per_class   = f1_score(base_labels,   base_preds,   average=None, labels=list(range(NUM_CLASSES)))
resnet_f1_per_class = f1_score(resnet_labels, resnet_preds, average=None, labels=list(range(NUM_CLASSES)))

x = np.arange(NUM_CLASSES)
width = 0.35

fig, ax = plt.subplots(figsize=(14, 6))
bars1 = ax.bar(x - width/2, base_f1_per_class,   width, label='Baseline CNN', color='#4C72B0', alpha=0.85)
bars2 = ax.bar(x + width/2, resnet_f1_per_class, width, label='ResNet18 FT',  color='#DD8452', alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels([CLASS_LABELS[c] for c in CLASS_NAMES], rotation=30, ha='right')
ax.set_ylabel('F1-score')
ax.set_title('Per-class F1-score: Baseline CNN vs ResNet18 Transfer Learning')
ax.set_ylim(0, 1.05)
ax.legend()
ax.grid(axis='y', alpha=0.3)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=8)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('f1_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# AUC-ROC (one-vs-rest)
@torch.no_grad()
def get_probabilities(model, loader):
    model.eval()
    all_probs, all_labels = [], []
    for imgs, labels in loader:
        imgs = imgs.to(DEVICE)
        logits = model(imgs)
        probs = F.softmax(logits, dim=1)
        all_probs.append(probs.cpu().numpy())
        all_labels.extend(labels.numpy())
    return np.vstack(all_probs), np.array(all_labels)

resnet_probs, true_labels_arr = get_probabilities(resnet_model, test_loader)

auc_per_class = roc_auc_score(
    true_labels_arr,
    resnet_probs,
    multi_class='ovr',
    average=None
)
macro_auc = roc_auc_score(true_labels_arr, resnet_probs, multi_class='ovr', average='macro')

print(f"\nAUC-ROC (macro): {macro_auc:.4f}")
print("\nAUC-ROC per class:")
for cls, auc in zip(CLASS_NAMES, auc_per_class):
    print(f"  {CLASS_LABELS[cls]:30s}: {auc:.4f}")

In [ ]:
# Summary table
summary_data = {
    'Model':         ['Baseline CNN', 'ResNet18 Transfer Learning'],
    'Test Accuracy': [f'{base_acc*100:.2f}%', f'{resnet_acc*100:.2f}%'],
    'Weighted F1':   [f'{base_f1:.4f}', f'{resnet_f1:.4f}'],
    'Macro AUC-ROC': ['—', f'{macro_auc:.4f}'],
    'Parameters':    [f'{sum(p.numel() for p in BaselineCNN().parameters()):,}',
                      f'{sum(p.numel() for p in models.resnet18().parameters()):,}'],
}
summary_df = pd.DataFrame(summary_data)
print("\n" + "=" * 70)
print(summary_df.to_string(index=False))
print("=" * 70)

## Conclusions

### Key findings

1. **Class imbalance** — `melanocytic nevus` accounts for ~67% of the dataset. Without the weighted sampler the model degrades to predicting only this class.

2. **Transfer Learning >> Scratch** — ResNet18 with ImageNet weights significantly outperforms the baseline CNN despite fewer training epochs. ImageNet features (textures, edges) transfer well to dermoscopic images.

3. **Melanoma vs. Nevus** — the most common misclassification pair. These two classes are visually similar, reflecting the real difficulty of dermatological diagnosis.

4. **Grad-CAM** — the model genuinely focuses on the lesion, not the background. For misclassified samples the attention is diffuse or targets irrelevant regions.

5. **Accuracy vs. F1** — due to class imbalance, high accuracy can be misleading. Weighted F1 and per-class AUC-ROC provide a more honest picture.

### Literature benchmarks
- ResNet18 ~80% accuracy
- VGG11 ~80.5% accuracy
- Our result: `**%` (fill in after training)

### Possible improvements
- EfficientNet-B3/B4 (higher capacity at similar parameter count)
- MixUp / CutMix augmentation
- Test-Time Augmentation (TTA)
- Focal Loss instead of CrossEntropyLoss